# 03 — SciPy: the Signals and Systems toolbox

This is the notebook that maps most directly onto your course. Nearly every named concept —
LTI systems, transfer functions, impulse and step response, filter design, the Laplace and
z-domains, spectral estimation — has a direct counterpart in `scipy.signal`.

If you have seen MATLAB's Signal Processing Toolbox, this is the same territory:
`tf` → `TransferFunction`, `step` → `signal.step`, `butter` → `signal.butter`,
`freqz` → `signal.freqz`, `filter` → `signal.lfilter`.

**Prerequisite:** notebooks 01 and 02. This one assumes you can build a time vector, take an
FFT, and read a Bode plot.

---

## Contents

| § | Topic |
|---|-------|
| 1 | Orientation: what lives in `scipy.signal` |
| 2 | Standard waveforms |
| 3 | Convolution and correlation, done fast |
| 4 | Continuous-time LTI systems |
| 5 | Discrete-time LTI systems |
| 6 | Frequency response: `freqs`, `freqz`, `bode` |
| 7 | IIR filter design |
| 8 | FIR filter design |
| 9 | `lfilter` vs `filtfilt` — causality and phase |
| 10 | Second-order sections and numerical stability |
| 11 | Windows and spectral leakage |
| 12 | Spectral estimation: periodogram, Welch, STFT |
| 13 | Resampling |
| 14 | The analytic signal, envelope, and instantaneous frequency |
| 15 | Peak finding |
| 16 | A worked case study: cleaning the ECG |
| 17 | Beyond `signal`: `scipy.fft`, `integrate`, `linalg` |
| 18 | Exercises |

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import signal

plt.rcParams.update({
    "figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
np.set_printoptions(precision=4, suppress=True)

print("scipy", scipy.__version__)

def time_vector(fs, duration):
    n = int(round(fs * duration))
    return np.arange(n) / fs, n

---
## 1. Orientation: what lives in `scipy.signal`

Rather than memorising the module, learn its six neighbourhoods:

| Neighbourhood | Representative functions |
|---------------|--------------------------|
| **Waveforms** | `chirp`, `square`, `sawtooth`, `gausspulse`, `unit_impulse`, `max_len_seq` |
| **Convolution** | `convolve`, `fftconvolve`, `oaconvolve`, `correlate`, `correlation_lags` |
| **LTI objects** | `lti`, `dlti`, `TransferFunction`, `ZerosPolesGain`, `StateSpace` |
| **Responses** | `impulse`, `step`, `lsim`, `dimpulse`, `dstep`, `dlsim`, `bode`, `freqresp` |
| **Filter design** | `butter`, `cheby1`, `cheby2`, `ellip`, `bessel`, `iirnotch`, `firwin`, `remez` |
| **Spectral** | `periodogram`, `welch`, `spectrogram`, `ShortTimeFFT`, `coherence`, `csd` |

Everything else is a variation on those.

---
## 2. Standard waveforms

Test signals you would otherwise hand-roll.

In [ ]:
fs = 1000.0
t, N = time_vector(fs, 2.0)

waves = {
    "square (50% duty)":  signal.square(2 * np.pi * 3 * t),
    "square (20% duty)":  signal.square(2 * np.pi * 3 * t, duty=0.2),
    "sawtooth":           signal.sawtooth(2 * np.pi * 3 * t),
    "triangle":           signal.sawtooth(2 * np.pi * 3 * t, width=0.5),
    "linear chirp 1→80":  signal.chirp(t, f0=1, f1=80, t1=2, method="linear"),
    "log chirp 1→80":     signal.chirp(t, f0=1, f1=80, t1=2, method="logarithmic"),
}

fig, ax = plt.subplots(3, 2, figsize=(12, 5.5), sharex=True)
for a, (name, w) in zip(ax.ravel(), waves.items()):
    a.plot(t, w, lw=0.8)
    a.set_title(name, fontsize=9)
for a in ax[-1]:
    a.set_xlabel("time [s]")
fig.tight_layout()

In [ ]:
# A Gaussian-modulated RF pulse -- the canonical radar/ultrasound test signal.
t_p = np.linspace(-1, 1, 2000, endpoint=False)
pulse, envelope = signal.gausspulse(t_p, fc=8, bw=0.5, retenv=True)

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(t_p, pulse, lw=0.9, label="gausspulse(fc=8, bw=0.5)")
ax.plot(t_p, envelope, "r--", lw=1.2, label="envelope")
ax.plot(t_p, -envelope, "r--", lw=1.2)
ax.set_xlabel("time [s]"); ax.legend()

# A unit impulse, which you will use constantly to probe systems.
print("unit_impulse(8, 3) =", signal.unit_impulse(8, 3))

---
## 3. Convolution and correlation, done fast

`np.convolve` is fine for short kernels. For long ones, use the FFT-based versions — the
complexity drops from $O(NM)$ to $O((N+M)\log(N+M))$.

In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(size=100_000)
h = rng.normal(size=2_000)

direct = np.convolve(x, h)
fast = signal.fftconvolve(x, h)
overlap = signal.oaconvolve(x, h)          # overlap-add: best when x >> h

print("results agree:", np.allclose(direct, fast, atol=1e-8),
      np.allclose(direct, overlap, atol=1e-8))
print()
print("np.convolve:")
%timeit -n 1 -r 2 np.convolve(x, h)
print("signal.fftconvolve:")
%timeit -n 3 -r 3 signal.fftconvolve(x, h)
print("signal.oaconvolve:")
%timeit -n 3 -r 3 signal.oaconvolve(x, h)

> **`signal.choose_conv_method`** picks for you if you would rather not think about it.

In [ ]:
print("for 100k x 2k :", signal.choose_conv_method(x, h))
print("for 100 x 5   :", signal.choose_conv_method(x[:100], h[:5]))

In [ ]:
# correlation_lags removes the off-by-one lag-axis arithmetic that trips everyone up.
fs = 1000.0
t, N = time_vector(fs, 1.0)
template = signal.gausspulse(np.linspace(-0.05, 0.05, 100), fc=80)
delay = 620

received = rng.normal(0, 0.7, N)
received[delay:delay + template.size] += template

corr = signal.correlate(received, template, mode="full")
lags = signal.correlation_lags(received.size, template.size, mode="full")
found = lags[np.argmax(corr)]

print(f"true delay {delay} samples, detected {found} samples")

fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(lags / fs, corr, lw=0.8)
ax.axvline(found / fs, color="r", ls="--", label=f"{found / fs * 1000:.0f} ms")
ax.set_xlabel("lag [s]"); ax.legend(); ax.set_title("Matched filter output")

---
## 4. Continuous-time LTI systems

A transfer function

$$H(s) = \frac{b_m s^m + \cdots + b_0}{a_n s^n + \cdots + a_0}$$

is entered as two coefficient lists, **highest power first**.

In [ ]:
# Second-order system: H(s) = wn^2 / (s^2 + 2*zeta*wn*s + wn^2)
wn, zeta = 5.0, 0.3
num = [wn ** 2]
den = [1, 2 * zeta * wn, wn ** 2]

sys = signal.TransferFunction(num, den)
print(sys)
print()
print("poles :", sys.poles)
print("zeros :", sys.zeros)
print("stable:", np.all(sys.poles.real < 0))

In [ ]:
t_imp, y_imp = signal.impulse(sys, N=800)
t_step, y_step = signal.step(sys, N=800)

# Arbitrary input via lsim.
t_in = np.linspace(0, 6, 1200)
u = signal.square(2 * np.pi * 0.5 * t_in)
t_out, y_out, _ = signal.lsim(sys, U=u, T=t_in)

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
ax[0].plot(t_imp, y_imp);  ax[0].set_title("impulse response h(t)")
ax[1].plot(t_step, y_step); ax[1].axhline(1, color="k", ls=":", lw=1)
ax[1].set_title("step response")
ax[2].plot(t_in, u, "0.7", lw=1, label="input")
ax[2].plot(t_out, y_out, label="output")
ax[2].set_title("response to a square wave"); ax[2].legend(fontsize=8)
for a in ax:
    a.set_xlabel("time [s]")
fig.tight_layout()

### Reading the step response: the standard performance metrics

Overshoot, rise time, settling time — the quantities every control-adjacent question asks for.

In [ ]:
def step_metrics(t, y, final=None, settle_band=0.02):
    """Overshoot %, peak time, rise time (10-90%), and 2% settling time."""
    final = y[-1] if final is None else final
    peak_i = int(np.argmax(y))
    overshoot = (y[peak_i] - final) / final * 100

    try:
        t_10 = t[np.argmax(y >= 0.1 * final)]
        t_90 = t[np.argmax(y >= 0.9 * final)]
        rise = t_90 - t_10
    except ValueError:
        rise = np.nan

    outside = np.where(np.abs(y - final) > settle_band * abs(final))[0]
    settle = t[outside[-1]] if outside.size else t[0]

    return {"overshoot_%": overshoot, "peak_time_s": t[peak_i],
            "rise_time_s": rise, "settling_time_s": settle, "final_value": final}

m = step_metrics(t_step, y_step)
for k, v in m.items():
    print(f"{k:<18} {v:.4f}")

# Theory check for a second-order system.
theory_os = 100 * np.exp(-np.pi * zeta / np.sqrt(1 - zeta ** 2))
print(f"\ntheoretical overshoot: {theory_os:.2f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(t_step, y_step, lw=1.6)
ax.axhline(1, color="k", ls=":", lw=1)
ax.axhspan(0.98, 1.02, color="tab:green", alpha=0.12)
ax.plot(m["peak_time_s"], y_step.max(), "ro")
ax.annotate(f"overshoot {m['overshoot_%']:.1f}%",
            xy=(m["peak_time_s"], y_step.max()), xytext=(m["peak_time_s"] + 0.5, y_step.max()),
            arrowprops=dict(arrowstyle="->"), fontsize=9)
ax.axvline(m["settling_time_s"], color="tab:purple", ls="--", lw=1)
ax.text(m["settling_time_s"] + 0.05, 0.2, f"settles at {m['settling_time_s']:.2f} s",
        fontsize=9, color="tab:purple")
ax.set_xlabel("time [s]"); ax.set_ylabel("y(t)")
ax.set_title(f"Step response metrics (ζ = {zeta}, ωₙ = {wn})")

### Three equivalent representations

Transfer function, zero-pole-gain, and state-space describe the same system. SciPy converts
between them freely, and different questions are easier in different forms.

In [ ]:
tf = signal.TransferFunction(num, den)
zpk = tf.to_zpk()
ss = tf.to_ss()

print("--- transfer function ---")
print("num:", tf.num, " den:", tf.den)
print("\n--- zero-pole-gain ---")
print("zeros:", zpk.zeros, " poles:", zpk.poles, " gain:", zpk.gain)
print("\n--- state space ---")
print("A =\n", ss.A, "\nB =\n", ss.B, "\nC =", ss.C, " D =", ss.D)
print("\neigenvalues of A == poles:", np.allclose(np.sort_complex(np.linalg.eigvals(ss.A)),
                                                  np.sort_complex(tf.poles)))

---
## 5. Discrete-time LTI systems

Same objects, with `dt` supplied. The difference equation

$$\sum_k a_k\, y[n-k] = \sum_k b_k\, x[n-k]$$

becomes `b` and `a` coefficient arrays, with `a[0]` normalised to 1.

In [ ]:
fs = 100.0
b = [0.2, 0.2]                     # y[n] = 0.2x[n] + 0.2x[n-1] + 0.6y[n-1]
a = [1.0, -0.6]

dsys = signal.dlti(b, a, dt=1 / fs)
print("poles:", dsys.poles, " |poles| =", np.abs(dsys.poles))
print("stable (all |p| < 1):", np.all(np.abs(dsys.poles) < 1))

n_imp, y_imp = signal.dimpulse(dsys, n=40)
n_step, y_step = signal.dstep(dsys, n=40)

fig, ax = plt.subplots(1, 2, figsize=(11, 2.8))
ax[0].stem(np.arange(40), y_imp[0].ravel(), basefmt=" "); ax[0].set_title("h[n]")
ax[1].stem(np.arange(40), y_step[0].ravel(), basefmt=" "); ax[1].set_title("step response")
for a_ in ax:
    a_.set_xlabel("n"); a_.axhline(0, color="k", lw=0.6)
fig.tight_layout()

> **`dimpulse` and `dstep` return a tuple of output arrays** (one per system output), so you
> need `y[0]` and often `.ravel()`. This trips up everyone once.

In [ ]:
# Running a discrete filter over real data: lfilter.
t, N = time_vector(fs, 3.0)
x = np.sin(2 * np.pi * 1.5 * t) + 0.4 * np.sin(2 * np.pi * 25 * t)
y = signal.lfilter(b, a, x)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, x, "0.7", lw=0.9, label="input (1.5 Hz + 25 Hz)")
ax.plot(t, y, lw=1.4, label="lfilter output")
ax.set_xlabel("time [s]"); ax.legend()
ax.set_title("The IIR low-pass removes the 25 Hz component")

---
## 6. Frequency response: `freqs`, `freqz`, `bode`

| function | domain | argument | returns |
|----------|--------|----------|---------|
| `signal.freqs(b, a, w)` | continuous (s) | rad/s | $H(j\omega)$ |
| `signal.freqz(b, a, fs=)` | discrete (z) | Hz if `fs` given | $H(e^{j\omega})$ |
| `signal.bode(sys)` | continuous | — | `(w, mag_dB, phase_deg)` |
| `signal.dbode(dsys)` | discrete | — | same |

In [ ]:
# Continuous: bode gives you dB and degrees directly.
w, mag_db, phase_deg = signal.bode(sys, n=1000)

fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].semilogx(w, mag_db); ax[0].set_ylabel("|H| [dB]")
ax[1].semilogx(w, phase_deg); ax[1].set_ylabel("phase [deg]")
ax[1].set_xlabel("ω [rad/s]")
for panel in ax:                     # note: not `a` -- that name holds our denominator
    panel.grid(True, which="both", alpha=0.3)
ax[0].axvline(wn, color="r", ls=":", lw=1)
ax[0].set_title(f"Bode plot, resonance at ωₙ = {wn} rad/s")
fig.tight_layout()

In [ ]:
# Discrete: freqz with fs= gives a frequency axis in Hz, which is what you want.
w_hz, H = signal.freqz(b, a, worN=2048, fs=fs)

fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].plot(w_hz, 20 * np.log10(np.abs(H)))
ax[0].set_ylabel("|H| [dB]"); ax[0].axhline(-3, color="r", ls=":", lw=1)
ax[1].plot(w_hz, np.degrees(np.unwrap(np.angle(H))))
ax[1].set_ylabel("phase [deg]"); ax[1].set_xlabel("frequency [Hz]")
ax[0].set_title(f"freqz -- axis runs to Nyquist = {fs/2:g} Hz")
fig.tight_layout()

> **Without `fs=`, `freqz` returns normalised frequency in radians/sample, running 0 to π.**
> π corresponds to Nyquist. Passing `fs=` and getting Hz back removes an entire class of
> off-by-a-factor-of-two errors — do it every time.

### Group delay

Group delay $\tau_g(\omega) = -\frac{d\phi}{d\omega}$ tells you how much each frequency is
delayed. Constant group delay = linear phase = no waveform distortion.

In [ ]:
b_iir, a_iir = signal.butter(6, 0.2)
b_fir = signal.firwin(61, 0.2)

fig, ax = plt.subplots(figsize=(9, 3))
for (bb, aa), name in [((b_iir, a_iir), "Butterworth IIR (order 6)"),
                       ((b_fir, [1.0]), "FIR (61 taps)")]:
    w_gd, gd = signal.group_delay((bb, aa), fs=2.0)
    ax.plot(w_gd, gd, label=name)
ax.set_xlabel("normalised frequency (1.0 = Nyquist)")
ax.set_ylabel("group delay [samples]")
ax.legend(); ax.set_title("FIR group delay is flat; IIR is not")

---
## 7. IIR filter design

The five classical families, and the trade-off each one makes. There is no best filter; there
is only which distortion you are willing to accept.

| family | passband | stopband | roll-off | phase |
|--------|----------|----------|----------|-------|
| **Butterworth** | maximally flat | monotonic | gentlest | poor |
| **Chebyshev I** | ripple | monotonic | steeper | poor |
| **Chebyshev II** | flat | ripple | steeper | poor |
| **Elliptic** | ripple | ripple | steepest | worst |
| **Bessel** | flat | monotonic | gentlest | **maximally linear** |

In [ ]:
fs = 1000.0
cutoff = 100.0
order = 6

designs = {
    "butter":  signal.butter(order, cutoff, fs=fs),
    "cheby1":  signal.cheby1(order, 1.0, cutoff, fs=fs),            # 1 dB passband ripple
    "cheby2":  signal.cheby2(order, 40.0, cutoff, fs=fs),           # 40 dB stopband
    "ellip":   signal.ellip(order, 1.0, 40.0, cutoff, fs=fs),
    "bessel":  signal.bessel(order, cutoff, fs=fs),
}

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
for name, (bb, aa) in designs.items():
    w, H = signal.freqz(bb, aa, worN=4096, fs=fs)
    ax[0].plot(w, 20 * np.log10(np.abs(H) + 1e-12), label=name)
    ax[1].plot(w, 20 * np.log10(np.abs(H) + 1e-12), label=name)

ax[0].set_ylim(-100, 5); ax[0].set_xlim(0, 500)
ax[0].set_title(f"Order-{order} low-pass, fc = {cutoff:g} Hz")
ax[1].set_ylim(-3, 0.5); ax[1].set_xlim(0, 130)
ax[1].set_title("passband detail -- see the ripple")
for a in ax:
    a.axvline(cutoff, color="k", ls=":", lw=1)
    a.axhline(-3, color="k", ls=":", lw=1)
    a.set_xlabel("frequency [Hz]"); a.set_ylabel("|H| [dB]"); a.legend(fontsize=8)
fig.tight_layout()

> **The passband panel is the one to study.** Butterworth and Bessel are flat. Chebyshev I and
> elliptic ripple by exactly the 1 dB you asked for. If your application measures amplitude,
> that ripple is a measurement error you have deliberately introduced in exchange for a
> steeper transition.

In [ ]:
# Filter types: low-pass, high-pass, band-pass, band-stop.
types = {
    "lowpass  (< 100 Hz)":     signal.butter(6, 100, btype="lowpass", fs=fs),
    "highpass (> 100 Hz)":     signal.butter(6, 100, btype="highpass", fs=fs),
    "bandpass (80-160 Hz)":    signal.butter(6, [80, 160], btype="bandpass", fs=fs),
    "bandstop (80-160 Hz)":    signal.butter(6, [80, 160], btype="bandstop", fs=fs),
}

fig, ax = plt.subplots(figsize=(10, 3.4))
for name, (bb, aa) in types.items():
    w, H = signal.freqz(bb, aa, worN=4096, fs=fs)
    ax.plot(w, 20 * np.log10(np.abs(H) + 1e-12), label=name)
ax.set_ylim(-80, 5); ax.set_xlim(0, 400)
ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("|H| [dB]"); ax.legend(fontsize=8)

### Letting the specification pick the order

Rather than guessing an order, state your requirements and let `buttord` / `cheb1ord` /
`ellipord` compute the minimum order that meets them.

In [ ]:
wp, ws = 100.0, 150.0     # passband edge, stopband edge (Hz)
gpass, gstop = 1.0, 60.0  # max passband loss (dB), min stopband attenuation (dB)

for name, ordfunc, design in [
    ("butter", signal.buttord, signal.butter),
    ("cheby1", signal.cheb1ord, lambda n, wc, **k: signal.cheby1(n, gpass, wc, **k)),
    ("ellip",  signal.ellipord, lambda n, wc, **k: signal.ellip(n, gpass, gstop, wc, **k)),
]:
    n, wc = ordfunc(wp, ws, gpass, gstop, fs=fs)
    print(f"{name:<8} needs order {n:>2}   (natural frequency {np.atleast_1d(wc)[0]:.1f} Hz)")

> **Elliptic gets there with the lowest order every time** — that is its entire purpose.
> The cost is ripple in *both* bands and badly non-linear phase. When you see an elliptic
> filter in a design, someone was fighting for computational budget.

---
## 8. FIR filter design

FIR filters have no feedback: $y[n] = \sum_k b_k x[n-k]$. They are unconditionally stable and
can have exactly linear phase. The price is many more coefficients for the same selectivity.

In [ ]:
numtaps = 101
fir_designs = {
    "firwin (Hamming)":  signal.firwin(numtaps, 100, fs=fs),
    "firwin (Blackman)": signal.firwin(numtaps, 100, window="blackman", fs=fs),
    "firwin (Kaiser β=8)": signal.firwin(numtaps, 100, window=("kaiser", 8.0), fs=fs),
    "remez (equiripple)": signal.remez(numtaps, [0, 90, 130, fs / 2], [1, 0], fs=fs),
}

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
for name, bb in fir_designs.items():
    w, H = signal.freqz(bb, worN=4096, fs=fs)
    ax[0].plot(w, 20 * np.log10(np.abs(H) + 1e-12), label=name)
    ax[1].plot(bb, lw=1, label=name)

ax[0].set_ylim(-110, 5); ax[0].set_xlim(0, 400)
ax[0].set_xlabel("frequency [Hz]"); ax[0].set_ylabel("|H| [dB]")
ax[0].set_title(f"{numtaps}-tap FIR low-pass"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("tap index"); ax[1].set_title("impulse responses (the coefficients)")
fig.tight_layout()

> **`remez` (Parks-McClellan) is optimal** in the minimax sense: for a given number of taps it
> achieves the smallest possible maximum error. The equiripple stopband in the plot is the
> signature. `firwin` is simpler to specify and usually good enough.

### Linear phase, made visible

An FIR filter with symmetric coefficients has exactly linear phase — a pure delay of
$(N-1)/2$ samples, identical at every frequency. No waveform distortion, only a shift.

In [ ]:
b_fir = signal.firwin(101, 100, fs=fs)
print("coefficients are symmetric:", np.allclose(b_fir, b_fir[::-1]))
print("expected delay: (101-1)/2 =", (101 - 1) / 2, "samples")

w, H = signal.freqz(b_fir, worN=2048, fs=fs)
phase = np.unwrap(np.angle(H))

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(w, phase)
ax[0].set_xlabel("frequency [Hz]"); ax[0].set_ylabel("phase [rad]")
ax[0].set_title("phase is a straight line in the passband")
w_gd, gd = signal.group_delay((b_fir, 1), fs=fs)
ax[1].plot(w_gd, gd); ax[1].axhline(50, color="r", ls=":", lw=1)
ax[1].set_xlabel("frequency [Hz]"); ax[1].set_ylabel("group delay [samples]")
ax[1].set_title("group delay is constant at 50 samples")
fig.tight_layout()

---
## 9. `lfilter` vs `filtfilt` — causality and phase

- **`lfilter`** runs the filter once, forward. This is what real hardware does. It introduces
  delay and phase distortion.
- **`filtfilt`** runs it forward, then backward. The phase cancels exactly — **zero phase
  distortion** — but the filter is non-causal (it uses future samples) and the effective
  magnitude response is squared.

Use `filtfilt` for offline analysis of recorded data. Use `lfilter` when the filter must run
in real time.

In [ ]:
fs = 500.0
t, N = time_vector(fs, 1.0)
rng = np.random.default_rng(2)
clean = signal.square(2 * np.pi * 3 * t)
noisy = clean + rng.normal(0, 0.25, N)

b, a = signal.butter(4, 20, fs=fs)
y_lfilter = signal.lfilter(b, a, noisy)
y_filtfilt = signal.filtfilt(b, a, noisy)

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(t, noisy, "0.85", lw=0.7, label="noisy")
ax.plot(t, clean, "k--", lw=1, label="clean")
ax.plot(t, y_lfilter, label="lfilter (delayed)")
ax.plot(t, y_filtfilt, label="filtfilt (aligned)")
ax.set_xlim(0.15, 0.55); ax.set_xlabel("time [s]"); ax.legend(fontsize=8, ncol=4)
ax.set_title("The edge positions tell you everything")

print(f"lfilter  RMS error vs clean: {np.sqrt(np.mean((y_lfilter - clean)**2)):.4f}")
print(f"filtfilt RMS error vs clean: {np.sqrt(np.mean((y_filtfilt - clean)**2)):.4f}")

> **Watch the transitions.** The `lfilter` trace lags the true edges; `filtfilt` sits on top
> of them. That is why the RMS error differs so much even though both filters have the same
> magnitude response.
>
> **The catch nobody mentions:** `filtfilt` applies the filter twice, so the effective
> attenuation is doubled in dB and the effective $-3$ dB point moves. If you need a precise
> cutoff, design for it.

---
## 10. Second-order sections and numerical stability

For orders above roughly 8, transfer-function coefficients (`b, a`) lose precision badly —
the polynomial roots become extremely sensitive to coefficient rounding. **Second-order
sections** (`sos`) factor the filter into cascaded biquads and are numerically robust.

**Rule: use `output='sos'` for any filter of order 8 or higher. There is no downside.**

In [ ]:
order = 16
b_ba, a_ba = signal.butter(order, 100, fs=1000, output="ba")
sos = signal.butter(order, 100, fs=1000, output="sos")

w1, H1 = signal.freqz(b_ba, a_ba, worN=4096, fs=1000)
w2, H2 = signal.sosfreqz(sos, worN=4096, fs=1000)

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(w1, 20 * np.log10(np.abs(H1) + 1e-30), label="ba (transfer function)")
ax.plot(w2, 20 * np.log10(np.abs(H2) + 1e-30), "--", label="sos (second-order sections)")
ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("|H| [dB]")
ax.set_title(f"Order-{order} Butterworth: 'ba' has visibly fallen apart")
ax.legend()

print("sos shape:", sos.shape, "-> 8 biquad sections, each a row of [b0 b1 b2 a0 a1 a2]")
print("\nfirst section:\n", sos[0])

> **Look at the stopband.** The `ba` curve stops descending and turns into numerical noise;
> the `sos` curve keeps going. This is not a plotting artefact — filtering real data with
> those `ba` coefficients would produce exactly that broken response.

The corresponding filtering functions are `sosfilt` and `sosfiltfilt`.

In [ ]:
t, N = time_vector(1000, 0.5)
x = np.sin(2 * np.pi * 30 * t) + np.sin(2 * np.pi * 300 * t)

y_bad = signal.lfilter(b_ba, a_ba, x)
y_good = signal.sosfilt(sos, x)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, y_bad, label="lfilter with 'ba' (unusable)", lw=1)
ax.plot(t, y_good, label="sosfilt with 'sos' (correct)", lw=1.4)
ax.set_xlabel("time [s]"); ax.legend()

---
## 11. Windows and spectral leakage

A window tapers the ends of your capture so the periodic extension has no discontinuity. The
trade is always the same: **narrower main lobe (better resolution) vs. lower sidelobes
(better dynamic range).** You cannot have both.

In [ ]:
M = 128
window_names = ["boxcar", "hann", "hamming", "blackman", ("kaiser", 8.0), "flattop"]

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
for wname in window_names:
    win = signal.get_window(wname, M)
    label = wname if isinstance(wname, str) else f"kaiser β={wname[1]}"
    ax[0].plot(win, label=label)

    W = np.abs(np.fft.rfft(win, 4096))
    W_db = 20 * np.log10(W / W.max() + 1e-12)
    ax[1].plot(np.linspace(0, M / 2, W_db.size), W_db, label=label)

ax[0].set_title("window shapes"); ax[0].set_xlabel("sample"); ax[0].legend(fontsize=7)
ax[1].set_xlim(0, 12); ax[1].set_ylim(-140, 5)
ax[1].set_title("their spectra (main lobe width vs sidelobe level)")
ax[1].set_xlabel("bins"); ax[1].set_ylabel("dB"); ax[1].legend(fontsize=7)
fig.tight_layout()

| window | main lobe | peak sidelobe | when to use |
|--------|-----------|---------------|-------------|
| `boxcar` (none) | narrowest | −13 dB | never, for real data |
| `hann` | 4 bins | −31 dB | the sensible default |
| `hamming` | 4 bins | −43 dB | slightly better sidelobes |
| `blackman` | 6 bins | −58 dB | when dynamic range matters |
| `kaiser(β)` | tunable | tunable | when you want to dial the trade-off |
| `flattop` | widest | −93 dB | **amplitude accuracy** — measuring peak height |

Practical resolution: **Hann unless you have a reason.** `flattop` when you need to read an
amplitude off the plot correctly, because it has almost no scalloping loss.

In [ ]:
# Two tones 60 dB apart, close in frequency: the window decides whether you see the weak one.
fs, N = 1000.0, 1024
t = np.arange(N) / fs
x = np.sin(2 * np.pi * 100 * t) + 1e-3 * np.sin(2 * np.pi * 115 * t)
f = np.fft.rfftfreq(N, 1 / fs)

fig, ax = plt.subplots(figsize=(10, 3.4))
for wname in ["boxcar", "hann", "blackman"]:
    win = signal.get_window(wname, N)
    X = np.abs(np.fft.rfft(x * win))
    ax.plot(f, 20 * np.log10(X / X.max() + 1e-15), label=wname, lw=1)
ax.axvline(115, color="r", ls=":", lw=1)
ax.text(117, -20, "weak tone, 60 dB down", color="r", fontsize=9)
ax.set_xlim(80, 140); ax.set_ylim(-120, 5)
ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("dB"); ax.legend()

---
## 12. Spectral estimation: periodogram, Welch, STFT

A single FFT of a noisy signal is a *very* noisy estimate of its spectrum — and, awkwardly,
it does not get less noisy as you collect more data. **Welch's method** fixes this: split the
record into overlapping segments, take the periodogram of each, and average.

In [ ]:
fs = 1000.0
t, N = time_vector(fs, 10.0)
rng = np.random.default_rng(3)
x = np.sin(2 * np.pi * 120 * t) + 2 * rng.normal(size=N)

f_p, P_p = signal.periodogram(x, fs, window="hann")
f_w, P_w = signal.welch(x, fs, nperseg=1024, noverlap=512)

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.semilogy(f_p, P_p, lw=0.4, alpha=0.6, label="periodogram (one long FFT)")
ax.semilogy(f_w, P_w, lw=1.5, label="Welch (averaged segments)")
ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("PSD [V²/Hz]")
ax.legend(); ax.set_title("Welch trades frequency resolution for a stable estimate")
ax.set_xlim(0, 500)

> **The trade-off is explicit in `nperseg`.** Longer segments → finer frequency resolution but
> fewer of them to average, so a noisier estimate. Shorter segments → smoother estimate,
> coarser resolution. There is no free lunch; there is only your choice of where to spend the
> data.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
for nperseg in [128, 512, 4096]:
    f_w, P_w = signal.welch(x, fs, nperseg=nperseg)
    ax.semilogy(f_w, P_w, lw=1, label=f"nperseg = {nperseg}  (Δf = {fs/nperseg:.2f} Hz)")
ax.set_xlim(80, 160); ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("PSD")
ax.legend(); ax.set_title("Resolution vs. variance")

In [ ]:
# Coherence: how linearly related are two signals, frequency by frequency?
b, a = signal.butter(4, 80, fs=fs)
y = signal.lfilter(b, a, x) + 0.5 * rng.normal(size=N)

f_c, Cxy = signal.coherence(x, y, fs, nperseg=1024)

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(f_c, Cxy)
ax.axvline(80, color="r", ls=":", lw=1, label="filter cutoff")
ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("coherence"); ax.set_ylim(0, 1.05)
ax.set_xlim(0, 300); ax.legend()
ax.set_title("Coherence ≈ 1 where y is a filtered copy of x, ≈ 0 where it is just noise")

### Time-varying spectra: `ShortTimeFFT`

`signal.spectrogram` still works and is fine. Modern SciPy also offers `ShortTimeFFT`, which
is more explicit about windowing, hop size, and the exact time axis — and it can invert.

In [ ]:
fs = 4000.0
t, N = time_vector(fs, 3.0)
x = signal.chirp(t, f0=100, f1=1200, t1=3, method="quadratic")
x[int(1.2 * fs):int(1.6 * fs)] += 2 * np.sin(2 * np.pi * 600 * t[int(1.2 * fs):int(1.6 * fs)])

win = signal.get_window("hann", 256)
SFT = signal.ShortTimeFFT(win, hop=64, fs=fs, scale_to="magnitude")
Sx = SFT.stft(x)
Sx_db = 20 * np.log10(np.abs(Sx) + 1e-10)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(Sx_db, origin="lower", aspect="auto", cmap="magma",
               extent=SFT.extent(N), vmin=Sx_db.max() - 70, vmax=Sx_db.max())
ax.set_xlabel("time [s]"); ax.set_ylabel("frequency [Hz]")
ax.set_title("ShortTimeFFT: quadratic chirp with a burst at 600 Hz")
fig.colorbar(im, ax=ax, label="magnitude [dB]")
ax.set_ylim(0, 1600)

In [ ]:
# STFT is invertible: analyse, modify, resynthesise.
x_back = SFT.istft(Sx, k1=N)
print("reconstruction error (max abs):", np.max(np.abs(x - x_back[:N])))

---
## 13. Resampling

Three functions, three different situations. Choosing wrongly is a classic source of
artefacts.

| function | method | use when |
|----------|--------|----------|
| `resample` | FFT-based | signal is periodic / you want an arbitrary ratio and the edges do not matter |
| `resample_poly` | polyphase FIR | **the usual choice** — rational ratio, well-behaved edges |
| `decimate` | filter + downsample | integer downsampling only, includes the anti-alias filter |

In [ ]:
fs_in = 1000.0
t, N = time_vector(fs_in, 1.0)
x = np.sin(2 * np.pi * 5 * t) + 0.3 * np.sin(2 * np.pi * 120 * t)

x_fft = signal.resample(x, 500)                     # -> 500 Hz
x_poly = signal.resample_poly(x, up=1, down=2)      # -> 500 Hz
x_dec = signal.decimate(x, q=2)                     # -> 500 Hz
x_naive = x[::2]                                    # -> 500 Hz, NO anti-alias filter

t_out = np.arange(500) / 500.0

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(t, x, "0.8", lw=0.8, label="original @ 1000 Hz")
ax.plot(t_out, x_poly, lw=1.2, label="resample_poly")
ax.plot(t_out, x_naive, lw=1.0, ls="--", label="naive x[::2] -- aliased")
ax.set_xlim(0, 0.25); ax.set_xlabel("time [s]"); ax.legend(fontsize=8)
ax.set_title("Downsampling to 500 Hz: the 120 Hz component survives correctly only with a filter")

In [ ]:
# Upsampling to a non-integer ratio: 1000 Hz -> 1500 Hz is up=3, down=2.
x_up = signal.resample_poly(x, up=3, down=2)
print(f"{N} samples @ 1000 Hz  ->  {len(x_up)} samples @ 1500 Hz")

from math import gcd
def rational_ratio(fs_from, fs_to):
    g = gcd(int(fs_from), int(fs_to))
    return int(fs_to // g), int(fs_from // g)

for a_, b_ in [(44100, 48000), (48000, 44100), (1000, 250)]:
    up, down = rational_ratio(a_, b_)
    print(f"{a_} Hz -> {b_} Hz  :  up={up}, down={down}")

---
## 14. The analytic signal, envelope, and instantaneous frequency

The **Hilbert transform** builds the analytic signal $x_a(t) = x(t) + j\,\hat{x}(t)$, whose
magnitude is the envelope and whose phase derivative is the instantaneous frequency. This is
how AM demodulation and vibration envelope analysis work.

In [ ]:
fs = 2000.0
t, N = time_vector(fs, 1.0)

carrier = np.sin(2 * np.pi * 200 * t)
modulator = 1 + 0.7 * np.sin(2 * np.pi * 5 * t)
am = modulator * carrier

analytic = signal.hilbert(am)
envelope = np.abs(analytic)
inst_phase = np.unwrap(np.angle(analytic))
inst_freq = np.diff(inst_phase) / (2 * np.pi) * fs

fig, ax = plt.subplots(2, 1, figsize=(10, 5))
ax[0].plot(t, am, "0.7", lw=0.6, label="AM signal")
ax[0].plot(t, envelope, "r", lw=1.6, label="envelope (|hilbert|)")
ax[0].plot(t, modulator, "k--", lw=1, label="true modulator")
ax[0].legend(fontsize=8, ncol=3); ax[0].set_xlabel("time [s]")
ax[0].set_title("Envelope detection recovers the modulating signal exactly")

ax[1].plot(t[1:], inst_freq, lw=0.8)
ax[1].axhline(200, color="r", ls=":", label="carrier 200 Hz")
ax[1].set_ylim(150, 250); ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("inst. freq [Hz]")
ax[1].legend(fontsize=8)
fig.tight_layout()

In [ ]:
# On a chirp, instantaneous frequency traces the sweep.
x = signal.chirp(t, f0=50, f1=400, t1=1, method="linear")
inst_f = np.diff(np.unwrap(np.angle(signal.hilbert(x)))) / (2 * np.pi) * fs

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(t[1:], inst_f, lw=1, label="measured")
ax.plot(t, 50 + (400 - 50) * t, "r--", lw=1.2, label="true sweep")
ax.set_xlabel("time [s]"); ax.set_ylabel("frequency [Hz]"); ax.legend()
ax.set_title("Instantaneous frequency of a linear chirp")

> **Edge effects are real.** The Hilbert transform is computed via the FFT, so it assumes
> periodicity. The first and last few percent of `envelope` and `inst_freq` are unreliable —
> discard them rather than explaining them.

---
## 15. Peak finding

`find_peaks` with its constraint arguments replaces a surprising amount of hand-written
threshold logic.

In [ ]:
fs = 360.0
t, N = time_vector(fs, 6.0)
rng = np.random.default_rng(4)
beats = np.zeros(N)
beat_positions = np.arange(50, N, int(fs * 0.83))
beats[beat_positions] = 1.0
ecg = signal.convolve(beats, signal.windows.gaussian(31, 3), mode="same")
ecg += 0.08 * rng.normal(size=N) + 0.2 * np.sin(2 * np.pi * 0.3 * t)

peaks, props = signal.find_peaks(
    ecg,
    height=0.5,        # minimum absolute height
    distance=int(0.4 * fs),  # at least 0.4 s apart -> max 150 bpm
    prominence=0.3,    # must stand out from its surroundings
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, ecg, lw=0.8)
ax.plot(t[peaks], ecg[peaks], "rv", ms=8, label=f"{len(peaks)} peaks")
ax.set_xlabel("time [s]"); ax.legend()

intervals = np.diff(t[peaks])
print(f"mean R-R interval : {intervals.mean():.3f} s")
print(f"heart rate        : {60 / intervals.mean():.1f} bpm")
print(f"available properties: {list(props.keys())}")

> **`prominence` is the argument that matters.** `height` alone fails whenever there is
> baseline drift, because a small bump riding on a high baseline passes the height test.
> Prominence measures how far a peak rises above the surrounding terrain, which is what you
> actually mean by "a peak".

---
## 16. A worked case study: cleaning the ECG

Everything above, applied to `data/ecg_like.csv`. The recording has three problems:
**baseline wander** (~0.2 Hz), **50 Hz mains hum**, and **broadband noise**. Each gets a
different tool.

In [ ]:
raw = np.loadtxt("../data/ecg_like.csv", delimiter=",", skiprows=1)
t_ecg, x_ecg = raw[:, 0], raw[:, 1]
fs_ecg = round(1 / np.mean(np.diff(t_ecg)))
print(f"{len(x_ecg)} samples, fs = {fs_ecg} Hz, duration = {t_ecg[-1]:.1f} s")

f_w, P_w = signal.welch(x_ecg, fs_ecg, nperseg=2048)
fig, ax = plt.subplots(1, 2, figsize=(13, 3))
ax[0].plot(t_ecg[:1800], x_ecg[:1800], lw=0.8)
ax[0].set(xlabel="time [s]", ylabel="mV", title="raw signal")
ax[1].semilogy(f_w, P_w)
ax[1].set(xlabel="frequency [Hz]", ylabel="PSD", title="raw spectrum")
ax[1].axvline(50, color="r", ls=":", label="50 Hz mains"); ax[1].legend(fontsize=8)
fig.tight_layout()

In [ ]:
# Step 1 -- remove baseline wander with a high-pass at 0.5 Hz.
sos_hp = signal.butter(4, 0.5, btype="highpass", fs=fs_ecg, output="sos")
x1 = signal.sosfiltfilt(sos_hp, x_ecg)

# Step 2 -- notch out the 50 Hz mains hum. Q sets how narrow the notch is.
b_notch, a_notch = signal.iirnotch(w0=50.0, Q=30.0, fs=fs_ecg)
x2 = signal.filtfilt(b_notch, a_notch, x1)

# Step 3 -- low-pass at 40 Hz to remove the remaining broadband noise.
sos_lp = signal.butter(6, 40, btype="lowpass", fs=fs_ecg, output="sos")
x3 = signal.sosfiltfilt(sos_lp, x2)

stages = [("raw", x_ecg), ("+ highpass 0.5 Hz", x1),
          ("+ 50 Hz notch", x2), ("+ lowpass 40 Hz", x3)]

fig, ax = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
for a, (name, sig_) in zip(ax, stages):
    a.plot(t_ecg, sig_, lw=0.8)
    a.set_ylabel(name, fontsize=8)
ax[-1].set_xlabel("time [s]"); ax[0].set_xlim(2, 7)
fig.suptitle("Progressive cleaning of the ECG", y=1.0)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
for name, sig_ in stages:
    f_w, P_w = signal.welch(sig_, fs_ecg, nperseg=2048)
    ax.semilogy(f_w, P_w, lw=1, label=name)
ax.set(xlabel="frequency [Hz]", ylabel="PSD", xlim=(0, 100))
ax.legend(fontsize=8)
ax.set_title("Each stage removes one identifiable component")

In [ ]:
# Now the beat detection works cleanly.
peaks, _ = signal.find_peaks(x3, prominence=0.4, distance=int(0.4 * fs_ecg))
rr = np.diff(t_ecg[peaks])

fig, ax = plt.subplots(2, 1, figsize=(10, 4.5))
ax[0].plot(t_ecg, x3, lw=0.8)
ax[0].plot(t_ecg[peaks], x3[peaks], "rv", ms=6)
ax[0].set(xlabel="time [s]", ylabel="mV", title=f"{len(peaks)} R-peaks detected")
ax[1].plot(t_ecg[peaks][1:], 60 / rr, "o-", ms=4)
ax[1].set(xlabel="time [s]", ylabel="instantaneous HR [bpm]",
          title="Heart rate variability (the sinusoid is the respiratory modulation)")
fig.tight_layout()

print(f"mean HR {60 / rr.mean():.1f} bpm, SD of R-R intervals {rr.std() * 1000:.1f} ms")

> **Note the order of operations and the choice of `filtfilt` throughout.** This is offline
> analysis of a recording, so zero-phase filtering is both available and correct — a
> phase-shifted R-peak would corrupt the interval measurement. In a real-time monitor you
> would use `sosfilt` and accept the delay.

---
## 17. Beyond `signal`: `scipy.fft`, `integrate`, `linalg`

### `scipy.fft` — a faster drop-in for `numpy.fft`

Same API, better performance, plus the DCT/DST which `numpy.fft` lacks.

In [ ]:
from scipy import fft as sfft

x = np.random.default_rng(5).normal(size=2 ** 16)
print("numpy.fft.rfft:")
%timeit -n 5 -r 3 np.fft.rfft(x)
print("scipy.fft.rfft:")
%timeit -n 5 -r 3 sfft.rfft(x)
print("\nidentical:", np.allclose(np.fft.rfft(x), sfft.rfft(x)))

# The DCT is the basis of JPEG and MP3 -- it concentrates energy in few coefficients.
sig_dct = sfft.dct(np.sin(2 * np.pi * 5 * np.linspace(0, 1, 256)), norm="ortho")
print("\nDCT: fraction of energy in the top 5 coefficients:",
      round(np.sort(sig_dct**2)[-5:].sum() / np.sum(sig_dct**2), 4))

### `scipy.integrate` — simulating a system from its differential equation

Sometimes you want to solve the ODE directly rather than use the transfer function, especially
for non-linear systems where no transfer function exists.

In [ ]:
from scipy.integrate import solve_ivp

wn, zeta = 5.0, 0.3

def second_order(t, state, u):
    """State-space form of y'' + 2*zeta*wn*y' + wn^2*y = wn^2*u."""
    y, ydot = state
    return [ydot, wn ** 2 * u(t) - 2 * zeta * wn * ydot - wn ** 2 * y]

t_span = (0, 5)
t_eval = np.linspace(*t_span, 1000)
sol = solve_ivp(second_order, t_span, [0, 0], t_eval=t_eval, args=(lambda t: 1.0,))

# Compare against the LTI step response.
sys = signal.TransferFunction([wn ** 2], [1, 2 * zeta * wn, wn ** 2])
t_lti, y_lti = signal.step(sys, T=t_eval)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(sol.t, sol.y[0], lw=2, label="solve_ivp (numerical ODE)")
ax.plot(t_lti, y_lti, "r--", lw=1.2, label="signal.step (LTI)")
ax.set_xlabel("time [s]"); ax.legend()
print("max difference:", np.max(np.abs(sol.y[0] - y_lti)))

In [ ]:
# A non-linear system, where the LTI machinery simply does not apply: the Van der Pol oscillator.
def van_der_pol(t, state, mu):
    x, v = state
    return [v, mu * (1 - x ** 2) * v - x]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
for mu in [0.5, 2.0, 4.0]:
    s = solve_ivp(van_der_pol, (0, 40), [1.0, 0.0], t_eval=np.linspace(0, 40, 4000), args=(mu,))
    ax[0].plot(s.t, s.y[0], lw=1, label=f"μ = {mu}")
    ax[1].plot(s.y[0], s.y[1], lw=1, label=f"μ = {mu}")
ax[0].set(xlabel="time", ylabel="x(t)", title="Van der Pol: self-sustaining oscillation")
ax[1].set(xlabel="x", ylabel="dx/dt", title="phase portrait -- a limit cycle")
for a in ax: a.legend(fontsize=8)
fig.tight_layout()

### `scipy.linalg` — the state-space workhorse

`scipy.linalg` extends `numpy.linalg` with matrix functions you need for state-space analysis,
notably `expm` (the matrix exponential) which gives the state-transition matrix.

In [ ]:
from scipy.linalg import expm

A = np.array([[0.0, 1.0], [-25.0, -3.0]])
B = np.array([[0.0], [25.0]])
C = np.array([[1.0, 0.0]])
D = np.array([[0.0]])

# State transition: x(t) = e^{At} x(0)
for tau in [0.0, 0.1, 0.5]:
    print(f"e^(A·{tau}) =\n{expm(A * tau)}\n")

# Discretise the continuous system at 100 Hz.
sys_c = signal.StateSpace(A, B, C, D)
sys_d = sys_c.to_discrete(dt=0.01)
print("discrete A (== expm(A*dt)):\n", sys_d.A)
print("matches expm:", np.allclose(sys_d.A, expm(A * 0.01)))

---
## 18. Exercises

**1. Design to a specification.** A sensor outputs signal content below 40 Hz. Sampling is at
1 kHz. Design a low-pass with less than 0.5 dB ripple below 40 Hz and at least 50 dB
attenuation above 60 Hz. Do it with each of `butter`, `cheby1`, and `ellip`; report the
required order and the group delay for each. Which would you ship, and why?

**2. `lfilter` vs `filtfilt`, quantified.** Take the square wave from §9 and measure the actual
delay introduced by `lfilter` (use cross-correlation). Confirm it equals the group delay at
low frequency. Show `filtfilt`'s delay is zero.

**3. Reconstruct the ECG pipeline as a function.** Write `clean_ecg(x, fs)` returning the
cleaned signal, with the three filter stages parameterised. Test it against
`data/ecg_like.csv` and report SNR improvement using the R-peak amplitude as signal and the
inter-beat segment variance as noise.

**4. Notch filter Q.** Sweep the `Q` parameter of `iirnotch` from 1 to 100 for a 50 Hz notch.
Plot the resulting notch width against Q and find the value where the notch starts removing
meaningful ECG content (hint: compare the cleaned R-peak shape).

**5. Aliasing on purpose.** Downsample the `accelerometer.csv` z-axis from 100 Hz to 20 Hz two
ways: `x[::5]` and `signal.decimate(x, 5)`. The data contains a 24 Hz burst. Show where the
burst energy ends up in each case, and explain the frequency you observe.

**6. STFT round-trip with modification.** Use `ShortTimeFFT` on a chirp, zero out all bins
above 500 Hz, and invert. Compare against applying a 500 Hz low-pass filter directly. Where do
they differ, and why?

**7. Implement `filtfilt` yourself.** Using only `lfilter` and array reversal, reproduce
`signal.filtfilt` for a simple filter. Compare against the real thing — the difference will be
at the edges, which is where `filtfilt`'s `padtype` machinery earns its keep.

**8. From difference equation to everything.** Given
$y[n] = 0.5y[n-1] - 0.2y[n-2] + x[n] + 0.3x[n-1]$: find the poles and zeros, plot the
pole-zero map, determine stability, compute and plot $h[n]$, plot $|H(e^{j\omega})|$, and
verify by filtering a chirp and comparing the output envelope against the magnitude response.

**9. Matched filter in noise.** Bury a 100-sample chirp in white noise at −20 dB SNR. Use
`signal.correlate` to detect it. Sweep the SNR and plot detection probability (over many
trials) against SNR. This is the ROC curve that underpins radar design.

---

### Quick reference

| Task | Call |
|------|------|
| transfer function | `signal.TransferFunction(num, den)` |
| discrete system | `signal.dlti(b, a, dt=1/fs)` |
| impulse / step | `signal.impulse(sys)`, `signal.step(sys)` |
| arbitrary input | `signal.lsim(sys, U, T)` / `signal.dlsim` |
| continuous response | `signal.bode(sys)`, `signal.freqs(b, a, w)` |
| discrete response | `signal.freqz(b, a, fs=fs)`, `signal.sosfreqz(sos, fs=fs)` |
| IIR design | `signal.butter(N, Wn, btype=, fs=, output="sos")` |
| minimum order | `signal.buttord(wp, ws, gpass, gstop, fs=)` |
| FIR design | `signal.firwin(numtaps, cutoff, fs=)`, `signal.remez(...)` |
| notch | `signal.iirnotch(w0, Q, fs=)` |
| apply (causal) | `signal.sosfilt(sos, x)` / `signal.lfilter(b, a, x)` |
| apply (zero-phase) | `signal.sosfiltfilt(sos, x)` / `signal.filtfilt(b, a, x)` |
| PSD | `signal.welch(x, fs, nperseg=)` |
| spectrogram | `signal.ShortTimeFFT(win, hop, fs)` → `.stft(x)` |
| resample | `signal.resample_poly(x, up, down)` |
| envelope | `np.abs(signal.hilbert(x))` |
| peaks | `signal.find_peaks(x, prominence=, distance=)` |
| group delay | `signal.group_delay((b, a), fs=)` |

### A short list of the mistakes that cost the most marks

1. Forgetting `fs=` in `freqz` and reading normalised radians as Hz.
2. Using `ba` output for a high-order filter and getting numerical rubbish.
3. Using `filtfilt` and then reporting the design cutoff, when the effective response is squared.
4. Downsampling by slicing without an anti-alias filter.
5. Not windowing before an FFT, then blaming leakage on the algorithm.
6. Reading `dimpulse` output without the `[0]` index.
7. Reporting `np.angle` phase without `np.unwrap`.